# Validate Storage Container
Current storage folder:
- storage/
-    |-----checkpoints/
-    |-----data/


In [1]:
import sys, os
from pathlib import Path
from src.dataset import load_cached_paths, download_dataset
from src.model import load_model

STORAGE_DIR = "./storage"

In [3]:
def valStorage():
    # Check exists
    if not Path(STORAGE_DIR).exists():
        print(f"Could not find storage at \"{STORAGE_DIR}\"")
        return
    else:
        print(f"Found storage at \"{STORAGE_DIR}\"")

    # Check dataset
    print(f"Checking for dataset...")
    dataset_paths = load_cached_paths()
    download = input(f"Found {len(dataset_paths)} downloaded scores. Download? (y/n)")
    if download == 'y':
        download_dataset(log=print, progress=lambda i, n: None, max_files=1000000)
        dataset_paths = load_cached_paths()
        print(f"Downloaded {len(dataset_paths)} scores.")

    print(f"Checking for checkpoints")
    checkpoint_dir = Path(STORAGE_DIR) / 'checkpoints'
    if not checkpoint_dir.exists():
        print("Could not find checkpoint path.")
    else:
        print("Found checkpoint folder. Scanning for infill model...")
        _, config = load_model()
        print(f"Found model config trained with {len(config.get('epoches'))} epoches")
        print(f"Found config: {config.__str__()}")


valStorage()

Found storage at "./storage"
Checking for dataset...
Fetching file tree from GitHub API...
Found 1462 score files.


Downloading: 100%|██████████| 1462/1462 [10:12<00:00,  2.39it/s]

Dataset cached: 1462 files in storage/data/openscore/
Downloaded 1462 scores.
Checking for checkpoints
Could not find checkpoint path.


In [4]:
import json, matplotlib.pyplot as plt
from pathlib import Path

cfg_path = Path(STORAGE_DIR) / "checkpoints/config.json"
if not cfg_path.exists():
    print("No config.json found — train the model first.")
else:
    epoches = json.loads(cfg_path.read_text()).get("epoches", [])
    if not epoches:
        print("No epoch data yet — run the training cell first.")
    else:
        train_losses = [e[0] for e in epoches]
        val_losses   = [e[1] for e in epoches]
        xs = range(1, len(epoches) + 1)

        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(xs, train_losses, marker="o", label="train loss")
        ax.plot(xs, val_losses,   marker="o", label="val loss")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.set_title("Training vs Validation Loss")
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

No config.json found — train the model first.
